## CS310 Natural Language Processing
## Lab 12: Training a Reasoning LLM with Reinforcement Learning

In this lab, we will practice training a reasoning LLM using the GRPO reinforcement learning algorithm.
The code is adapted from [reasoning-from-scratch](https://github.com/rasbt/reasoning-from-scratch).

In [1]:
import torch 
import copy
import json
from pprint import pprint

## T0. Loading Model and Data

We will use Qwen3-0.6B-Base as the model to be trained, and MATH-500 as the training data.

In [2]:
from utils import get_device, load_model_and_tokenizer

device = get_device()

from pathlib import Path

local_dir = Path("E:/CS310-Natural-Language-Processing/lab/lab12/qwen3")   # 绝对路径

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
    local_dir=local_dir,
)

Using NVIDIA CUDA GPU


In [3]:
from utils import render_prompt, generate_text_stream_concat_flex, generate_text_top_p_stream_cache

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)
print(prompt)

You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
Half the value of $3x-9$ is $x+37$. What is the value of $x$?

Answer:


In [4]:
torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

Next, load the MATH-500 dataset.

In [5]:
def load_math_train(file_path="math_full_minus_math500.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

math_train = load_math_train()
print("Dataset size:", len(math_train))

Dataset size: 12000


In [6]:
pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

## T1. Sampling Rollouts

In this section, we implement a function to sample multiple responses (rollouts) from the model for a given prompt.

Here we review the KV-cache style of generation a bit.

- Feed the prompt to the model to get the logits and initialize the cache.
- In each step of the generation loop, 
  - Compute the probability distribution of the next token using `torch.softmax()` on logits.
  - Filter the distribution using `top_p_filter()` to get the top-p filtered distribution.
  - Sample one token from the filtered distribution into `next_token`, using `torch.multinomial()`.
  - Computing the new logits by feeding `new_token` to the model.

In [7]:
from utils import KVCache, top_p_filter

@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
 ):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for _ in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        ### START YOUR CODE ###
        # Sample one token from the filtered distribution
        orig_device = logits.device
        next_token = torch.multinomial(probas.cpu(), num_samples=1).to(orig_device).squeeze(0)
        ### END YOUR CODE ###

        token_id = next_token.item()
        generated.append(token_id)

        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break
        ### START YOUR CODE ###
        # Feed the new token to the model to get the logits
        # `next_token` has shape [1], expand to [1,1] as batch sequence
        logits = model(next_token.unsqueeze(0), cache=cache)[:, -1]
        ### END YOUR CODE ###

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )

    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

In [8]:
raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(12)

# Use temperature=0.9, top_p=0.9
token_ids, prompt_len, answer_text = sample_response(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    device=device,
    max_new_tokens=512,
    temperature=0.9,
    top_p=0.9,
)

print(answer_text)

 \boxed{22}<|endoftext|>


You can change the seed to get different rollouts.

## T2. Compute Rewards

In practice, we would call `sample_response` multiple times to get a batch of rollouts.

For simplicity of the GRPO walkthrough, we will use a set of example rollouts here:

In [9]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

We will use two helper functions `extract_final_candiate()` and `grade_answer()`.

`extract_final_candidate()` uses regular expressions to extract the final candidate answer from an input rollout string.

In [10]:
from utils import extract_final_candidate

extracted = extract_final_candidate(rollouts[0])
print("extracted:", extracted)

extracted: 83


The second helper function `grade_answer()` computes the reward for a given extracted answer (from the rollout) and the ground truth answer.

In [11]:
from utils import grade_answer

extracted = extract_final_candidate(rollouts[0])
is_correct = grade_answer(extracted, "83")
print("is correct:", is_correct)

is correct: True


Combining the two helper functions, we can write the function `reward_rlvr()` to compute the reward for a given rollout.

It calls `extract_final_candidate()` first to extract an answer. If no answer can be extracted, i.e., there is no `\boxed{}` in the rollout, a reward of `0.0` is returned. 

Otherwise, it calls `grade_answer()` to compute the reward.

In [12]:
def reward_rlvr(answer_text, ground_truth):
    ### START YOUR CODE ###
    # Call `extract_final_candidate()` to extract an answer, specifying `fallback=None`
    extracted = extract_final_candidate(answer_text, fallback=None)

    if not extracted:
        return 0.0

    # Call `grade_answer()` to compute the reward
    correct = grade_answer(extracted, ground_truth)
    ### END YOUR CODE ###

    return float(correct)

In [13]:
# Test
rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward}\n")
    rollout_rewards.append(reward)

Answer: '\\boxed{83}'
Reward: 1.0

Answer: 'The correct answer is \\boxed{83}'
Reward: 1.0

Answer: 'The final answer is 83'
Reward: 0.0

Answer: 'We get \\boxed{38}'
Reward: 0.0



Given the reward scores, we can compute the `advantages` for each rollout.

The formula for the advantage is:

$$
A_i = \frac{r_i - \mu}{\sigma + \epsilon}
$$

where $r_i$ is the reward for the $i$-th rollout, $\mu$ is the mean of the rewards, and $\sigma$ is the standard deviation of the rewards.

In [14]:
rewards = torch.tensor(rollout_rewards, device=device)

### START YOUR CODE ###
epsilon = 1e-4
advantages = (rewards - rewards.mean()) / (rewards.std(unbiased=False) + epsilon)
### END YOUR CODE ###

print(rewards)
print(advantages)

tensor([1., 1., 0., 0.], device='cuda:0')
tensor([ 0.9998,  0.9998, -0.9998, -0.9998], device='cuda:0')


Note that if all rewards are identical, then the advantages are all 0.

It means the policy model will not be updated if all answers are correct or wrong. 

## T3. GRPO Objective

The formula for GRPO objective (to maximize) is:

$$
\frac{1}{G}\sum_{i=1}^{G}\min\left(\frac{\pi_\theta(o_i\mid q)}{\pi_{\text{ref}}(o_i\mid q)}A_i,\; \text{clip}\left(\frac{\pi_\theta(o_i\mid q)}{\pi_{\text{ref}}(o_i\mid q)}, 1-\varepsilon, 1+\varepsilon\right)A_i\right)
$$

Note:
- We omit the KL divergence term $-\beta D_{\text{KL}}(\pi_\theta\|\pi_{\text{ref}})$
- $A_i$ is the advantage computed in the previous section.
- Two models are involved: $\pi_\theta$ the policy model, and $\pi_{\text{ref}}$ the old policy model.
- In the ratio $\frac{\pi_\theta(o_i\mid q)}{\pi_{\text{ref}}(o_i\mid q)}$, $\pi_\theta(o_i\mid q)$ is the probability of the $i$-th rollout (product of the probabilities of all tokens) under the policy model. 
- In order to compute the *ratio* of two probabilities with good numerical stability, we compute the log-probabilities of each rollout (sum of the log-probabilities of all tokens) instead. 
- The fianl ratio is then calculated as:
  $\exp(\log \pi_\theta(o_i\mid q) - \log \pi_{\text{ref}}(o_i\mid q))$

So, we need to first define the function for computing the log-probabilities.

Hints:
- First compute the logits.
- Then use `torch.log_softmax()` to compute the log-probabilities.
- Collect the log-probabilities of target tokens, i.e., exclude the tokens from index 0 to `prompt_len - 1`.
- Return the sum of the log-probabilities of the target tokens.

In [15]:
def sequence_logprob(model, token_ids, prompt_len):
    ### START YOUR CODE ###
    # Handle edge cases: no generated tokens
    if token_ids.numel() <= prompt_len:
        return torch.tensor(0.0, device=token_ids.device)

    input_ids = token_ids.unsqueeze(0)
    logits = model(input_ids)  # [1, L, V]
    logprobs = torch.log_softmax(logits, dim=-1)[0]  # [L, V]

    # target token ids are input token ids shifted by 1
    # For causal LM, logits at position i predict token at i+1
    start = max(0, prompt_len - 1)
    end = token_ids.numel() - 1
    if end <= start:
        return torch.tensor(0.0, device=token_ids.device)

    targets = token_ids[prompt_len:]
    selected_logprobs = logprobs[start:end, targets]
    ### END YOUR CODE ###

    return torch.sum(selected_logprobs)

In [16]:
# Test
rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(
        model=model,
        token_ids=torch.tensor(token_ids, device=device),
        prompt_len=prompt_len,
    )
    print(f"Answer:  {text}")
    print(f"Logprob: {logprob.item():.4f}\n")
    rollout_logps.append(logprob)

Answer:  \boxed{83}
Logprob: -358.0000

Answer:  The correct answer is \boxed{83}
Logprob: -1216.0000

Answer:  The final answer is 83
Logprob: -516.0000

Answer:  We get \boxed{38}
Logprob: -760.0000



OK, next, we can implement the GRPO loss function.

- Input `model` is the policy model. `old_model` is the old policy model (optional).
- The first `for` loop is used to sample rollouts by calling `sample_response()`. After each rollout is sampled, compute the log-probability under the old policy by calling `sequence_logprob()` (i.e., $\pi_{\text{ref}}(o_i\mid q)$, stored in `old_logps`), and the reward by calling `reward_rlvr()`.
- After the first `for` loop, collect all rewards and compute the `advantages` (and detach it to `adv`).
- There is an additional step to check if the advantages are all 0. If so, the function will return early.
- The second `for` loop is used to compute the log-probabilities of the rollouts under the current policy, $\pi_\theta(o_i\mid q)$, stored in `new_logps`.
- Compute the `ratio`, defined as $\exp(\log \pi_\theta(o_i\mid q) - \log \pi_{\text{ref}}(o_i\mid q))$, and clip it to get `clipped_ratio`.
- Compute the clipped and unclipped advantages, by multiplying `ratio`(`clipped_ratio`) with `adv` (`advantages` detached).
- The GRPO objective computation is already implemented for you, by calling `torch.where()` conditioned on `adv >= 0`. The reason is that when `adv >= 0`, we use the smaller advantages as the objective, to avoid overly big positive updating. When `adv < 0`, we use the larger ones as the objective (closer to 0), to avoid overly big negative updating.
- Lastly, the loss is computed as the negative of the mean of GRPO objective.

In [17]:
def compute_grpo_loss(
    model,
    old_model,
    tokenizer,
    example,
    device,
    num_rollouts=4,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    clip_eps=10.0,
    skip_zero_adv=False,
):
    roll_old_logps, roll_rewards, samples = [], [], []
    roll_token_ids, roll_prompt_lens = [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training # save the training mode
    model.eval()
    if old_model is None:
        old_model = model

    # Sample rollouts using the old policy
    for _ in range(num_rollouts):
        ### START YOUR CODE ###
        # Call `sample_response()` 
        token_ids, prompt_len, text = sample_response(
            old_model, tokenizer, prompt, device,
            max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p,
        )
        ### END YOUR CODE ###

        # Compute the log-probability of the rollout under the old policy
        with torch.no_grad():
            ### START YOUR CODE ###
            old_logp = sequence_logprob(old_model, token_ids, prompt_len)
            ### END YOUR CODE ###

        # Compute the reward
        ### START YOUR CODE ###
        reward = reward_rlvr(answer_text=text, ground_truth=example.get("answer", ""))
        ### END YOUR CODE ###

        # Collect the rollout information
        roll_old_logps.append(old_logp)
        roll_rewards.append(reward)
        roll_token_ids.append(token_ids)
        roll_prompt_lens.append(prompt_len)
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training: # restore training mode
        model.train()

    # Compute the advantages using the rewards collected
    rewards = torch.tensor(roll_rewards, device=device)
    ### START YOUR CODE ###
    epsilon = 1e-4
    advantages = (rewards - rewards.mean()) / (rewards.std(unbiased=False) + epsilon)
    ### END YOUR CODE ###

    # Detech advantages to avoid gradient computation
    adv = advantages.detach()

    # Check if all advantages are 0
    is_zero_adv = torch.allclose(
        advantages,
        torch.zeros_like(advantages),
        atol=1e-8,
        rtol=0.0,
    )
    if skip_zero_adv and is_zero_adv:
        return {
            "loss": 0.0,
            "policy_ratio": 1.0,
            "rewards": roll_rewards,
            "advantages": advantages.detach().cpu().tolist(),
            "is_zero_adv": True,
            "samples": samples,
            "loss_tensor": None,
        }

    # Recompute sequence log-probs under the current policy (with grad)
    ### START YOUR CODE ###
    new_logps = []
    for token_ids, prompt_len in zip(roll_token_ids, roll_prompt_lens):
        new_logps.append(sequence_logprob(model, token_ids.to(device), prompt_len))
    ### END YOUR CODE ###
    new_logps = torch.stack(new_logps)

    old_logps = torch.stack(roll_old_logps).detach()

    # Compute the ratio, and clip it
    ### START YOUR CODE ###
    log_ratio = new_logps - old_logps
    ratio = torch.exp(log_ratio)
    clipped_ratio = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps)
    ### END YOUR CODE ###

    # Scale the adv using the ratio (clipped and unclipped)
    ### START YOUR CODE ###
    clipped = clipped_ratio * adv
    unclipped = ratio * adv
    ### END YOUR CODE ###

    # Compute the objective
    obj = torch.where(
        adv >= 0,
        torch.minimum(unclipped, clipped), # when A >= 0, use the smaller one as the objective, to avoid overly big positive updating.
        torch.maximum(unclipped, clipped), # when A < 0, use the larger one as the objective (closer to 0), to avoid overly big negative updating.
    )

    # Convert the objective to a loss
    loss = -obj.mean()

    return {
        "loss": loss.item(),
        "policy_ratio": ratio.mean().item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "is_zero_adv": is_zero_adv,
        "samples": samples,
        "loss_tensor": loss,
    }

In [18]:
# Test
torch.manual_seed(123)

stats = compute_grpo_loss(
    model=model,
    old_model=None,
    tokenizer=tokenizer,
    example=math_train[5],
    device=device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9
)

print("loss:", stats["loss"])
print("rewards:", stats["rewards"])
print("advantages:", stats["advantages"])
print("policy_ratio:", stats["policy_ratio"])

# You are expected to see the following output:
# loss: -0.0
# rewards: [0.0, 0.0]
# advantages: [0.0, 0.0]
# policy_ratio: 1.0

loss: -0.0
rewards: [0.0, 0.0]
advantages: [0.0, 0.0]
policy_ratio: 1.0


## T4. Training Loop

A training loop function is implemented for you. Run it with a small `steps` to see if it works.

In [19]:
from utils import save_checkpoint

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=4,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    clip_eps=10.0,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir="checkpoints",
):
    if steps is None:
        steps = len(math_data)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0

    try:
        for step in range(steps):
            current_step = step + 1
            example = math_data[step % len(math_data)]

            # Make copy of model from previous step as old_model
            old_model = copy.deepcopy(model).to(device)
            old_model.eval()
            for p in old_model.parameters():
                p.requires_grad = False

            stats = compute_grpo_loss(
                model=model,
                old_model=old_model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                clip_eps=clip_eps,
            )

            if stats["loss_tensor"] is not None:
                optimizer.zero_grad()
                stats["loss_tensor"].backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                # Check NaN in gradients
                has_nan_grad = any(p.grad is not None and p.grad.isnan().any().item() for p in model.parameters())
                if has_nan_grad:
                    print("WARNING: NaN detected in GRADIENTS before optimizer.step()!")

                optimizer.step()
            
            # Check NaN in parameters
            has_nan = any(p.isnan().any().item() for p in model.parameters())
            if has_nan:
                print("WARNING: Model weights contain NaN after optimizer step!")
                break

            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            advantage_tensor = torch.tensor(stats["advantages"])
            adv_avg = advantage_tensor.mean().item()
            adv_std = advantage_tensor.std().item()
            step_tokens = sum(sample["gen_len"] for sample in stats["samples"])
            avg_response_len = (
                step_tokens / len(stats["samples"]) if stats["samples"] else 0.0
            )

            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f} "
                f"adv_avg={adv_avg:.2f} "
                f"adv_std={adv_std:.2f} "
                f"policy_ratio={stats['policy_ratio']:.2f}"
            )

            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model

In [20]:
# Test
torch.manual_seed(123)

# Keep the original lower-precision weights to reduce memory use on GPU.
# If you still hit OOM, run this cell on CPU instead of CUDA.
train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=1,
    num_rollouts=1,
    max_new_tokens=32,
    temperature=0.8,
    top_p=0.9,
    clip_eps=10.0,
    lr=1e-5,
    checkpoint_every=0,
)

[Step 1/1] loss=-0.0000 reward_avg=0.000 avg_resp_len=3.0 adv_avg=0.00 adv_std=nan policy_ratio=1.00


C:\Users\mxm\AppData\Local\Temp\ipykernel_448\1465829341.py:70: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  adv_std = advantage_tensor.std().item()


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)